# HapticWay §5.6 — Custom Object Detection Model

Trains a **YOLOv8n** model (320×320, INT8) on 8 navigation-relevant classes:
`person`, `bicycle`, `bench`, `chair`, `door`, `staircase`, `pole`, `wet_floor_sign`.

**Why YOLOv8 instead of EfficientDet-Lite?**
TFLite Model Maker requires Python ≤3.10 and is deprecated. Ultralytics YOLOv8 works
on the current Colab Python (3.11+), is actively maintained, and achieves higher mAP
on the same hardware.

**Runtime required:** GPU (T4). Runtime → Change runtime type → T4 GPU.

**Time:** ~1.5 hours total (dataset download ~45 min, training ~45 min on T4).

**Outputs:**
- `hapticway_custom.tflite` — INT8 quantized YOLOv8n (320×320)
- `hapticway_labels.txt` — 9-line label file (??? + 8 classes)

After downloading, follow the integration steps in cell 8.

## 1 · Install dependencies

In [ ]:
!pip install -q ultralytics
!pip install -q fiftyone==0.23.8
!pip install -q pycocotools

In [ ]:
import os, json, shutil, yaml
from pathlib import Path
from ultralytics import YOLO

print('Ultralytics ready.')

## 2 · Download training data from Open Images v7

Uses Google's Open Images v7 dataset — no account needed.
Downloads bounding-box annotations for 7 of the 8 target classes.
`wet_floor_sign` is handled separately in §3.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

# Open Images class names → HapticWay app names
CLASS_MAP = {
    'Person':   'person',
    'Bicycle':  'bicycle',
    'Bench':    'bench',
    'Chair':    'chair',
    'Door':     'door',
    'Stairs':   'staircase',
    'Pole':     'pole',
}
OI_CLASSES = list(CLASS_MAP.keys())

N_TRAIN = 300   # images per class
N_VAL   = 60

print('Downloading training split...')
oi_train = foz.load_zoo_dataset(
    'open-images-v7',
    split='train',
    label_types=['detections'],
    classes=OI_CLASSES,
    max_samples=N_TRAIN * len(OI_CLASSES),
    dataset_name='hapticway_oi_train',
)
print(f'Train samples: {len(oi_train)}')

print('Downloading validation split...')
oi_val = foz.load_zoo_dataset(
    'open-images-v7',
    split='validation',
    label_types=['detections'],
    classes=OI_CLASSES,
    max_samples=N_VAL * len(OI_CLASSES),
    dataset_name='hapticway_oi_val',
)
print(f'Val samples: {len(oi_val)}')

## 3 · Download wet_floor_sign data from Roboflow (optional)

Sign up at https://roboflow.com (free). Go to
https://universe.roboflow.com and search **"wet floor sign detection"**.
Pick a dataset with ≥200 images, click **Download → COCO format**,
copy the download snippet and paste it below.

If you skip this cell, the model will not detect `wet_floor_sign`.
The app's `_targetClasses` filter will simply never match that label.

In [ ]:
# OPTIONAL — uncomment and fill in your Roboflow snippet:
#
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_API_KEY')
# project = rf.workspace('YOUR_WORKSPACE').project('wet-floor-sign-detection')
# version = project.version(1)
# rf_dataset = version.download('coco', location='/content/rf_wfs')
#
# After downloading, set:
# WFS_TRAIN_DIR  = '/content/rf_wfs/train'
# WFS_TRAIN_JSON = '/content/rf_wfs/train/_annotations.coco.json'
# WFS_VAL_DIR    = '/content/rf_wfs/valid'
# WFS_VAL_JSON   = '/content/rf_wfs/valid/_annotations.coco.json'
# HAVE_WFS = True

HAVE_WFS = False  # set True after uncommenting above

## 4 · Export to COCO JSON and remap class names

In [ ]:
DATA_DIR = Path('/content/hapticway_data')

# Class list in fixed order — this order defines the class indices (0-based) in the model
CLASSES = ['person', 'bicycle', 'bench', 'chair', 'door', 'staircase', 'pole']

def export_yolo(fo_dataset, split: str, class_map: dict, classes: list):
    """Export fiftyone dataset to YOLO format, remapping class names."""
    out_images = DATA_DIR / 'images' / split
    out_labels = DATA_DIR / 'labels' / split
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    for sample in fo_dataset.iter_samples(progress=True):
        # Copy image
        src = Path(sample.filepath)
        dst_img = out_images / src.name
        shutil.copy(str(src), str(dst_img))

        # Write YOLO label file
        w, h = sample.metadata.width, sample.metadata.height
        label_path = out_labels / (src.stem + '.txt')
        lines = []
        dets = sample.ground_truth
        if dets is None:
            label_path.write_text('')
            continue
        for det in dets.detections:
            oi_name = det.label
            app_name = class_map.get(oi_name)
            if app_name is None or app_name not in classes:
                continue
            cls_id = classes.index(app_name)
            # fiftyone bbox: [x, y, w, h] normalised, top-left origin
            bx, by, bw, bh = det.bounding_box
            cx = bx + bw / 2
            cy = by + bh / 2
            lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        label_path.write_text('\n'.join(lines))

print('Exporting training images...')
export_yolo(oi_train, 'train', CLASS_MAP, CLASSES)
print('Exporting validation images...')
export_yolo(oi_val,   'val',   CLASS_MAP, CLASSES)

# Write dataset.yaml for YOLOv8
dataset_yaml = {
    'path': str(DATA_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'nc':    len(CLASSES),
    'names': {i: name for i, name in enumerate(CLASSES)},
}
yaml_path = DATA_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print('dataset.yaml:')
print(yaml_path.read_text())

In [ ]:
# Merge wet_floor_sign images/labels into the YOLO dataset (if Roboflow data was downloaded)
if HAVE_WFS:
    WFS_CLASS_ID = len(CLASSES)  # wet_floor_sign gets the next class index
    CLASSES.append('wet_floor_sign')

    for split, rf_dir in [('train', WFS_TRAIN_DIR), ('val', WFS_VAL_DIR)]:
        rf_dir = Path(rf_dir)
        # Load COCO JSON
        anno_file = rf_dir / '_annotations.coco.json'
        with open(anno_file) as f:
            coco = json.load(f)

        # Build image_id → filename map
        id_to_file = {img['id']: img['file_name'] for img in coco['images']}

        # Group annotations by image
        from collections import defaultdict
        img_anns = defaultdict(list)
        for ann in coco['annotations']:
            img_anns[ann['image_id']].append(ann)

        dst_img = DATA_DIR / 'images' / split
        dst_lbl = DATA_DIR / 'labels' / split

        for img_meta in coco['images']:
            fname = img_meta['file_name']
            src_img = rf_dir / 'images' / fname
            if not src_img.exists():
                src_img = rf_dir / fname
            if src_img.exists():
                shutil.copy(str(src_img), str(dst_img / fname))

            iw, ih = img_meta['width'], img_meta['height']
            lines = []
            for ann in img_anns[img_meta['id']]:
                x, y, w, h = ann['bbox']  # COCO: top-left x,y + w,h (pixels)
                cx = (x + w / 2) / iw
                cy = (y + h / 2) / ih
                bw = w / iw
                bh = h / ih
                lines.append(f'{WFS_CLASS_ID} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

            stem = Path(fname).stem
            (dst_lbl / (stem + '.txt')).write_text('\n'.join(lines))

        print(f'Merged {len(coco["images"])} wet_floor_sign {split} images')

    # Update dataset.yaml with new class count
    dataset_yaml['nc'] = len(CLASSES)
    dataset_yaml['names'] = {i: name for i, name in enumerate(CLASSES)}
    with open(yaml_path, 'w') as f:
        yaml.dump(dataset_yaml, f, default_flow_style=False)
    print('dataset.yaml updated with wet_floor_sign')
else:
    print('Skipping wet_floor_sign merge (HAVE_WFS=False)')

## 5 · Train YOLOv8n

In [ ]:
# YOLOv8n: nano variant — fastest, smallest (~3 MB), sufficient for 320×320 mobile inference.
# Starting from COCO-pretrained weights (transfer learning).
model = YOLO('yolov8n.pt')

results = model.train(
    data=str(yaml_path),
    epochs=60,
    imgsz=320,
    batch=16,
    device=0,          # GPU
    name='hapticway',
    project='/content/runs',
    exist_ok=True,
    patience=15,       # early stop if no improvement for 15 epochs
    save=True,
    plots=True,
)
print('Training complete. Best weights:', results.save_dir)

## 6 · Evaluate

## 7 · Export INT8 TFLite

In [ ]:
# Load best checkpoint and evaluate on validation set
best = YOLO(f'/content/runs/hapticway/weights/best.pt')
metrics = best.val(data=str(yaml_path), imgsz=320)

print(f'mAP50:       {metrics.box.map50:.3f}')
print(f'mAP50-95:    {metrics.box.map:.3f}')
print('Per-class AP50:', {CLASSES[i]: f"{v:.3f}" for i, v in enumerate(metrics.box.ap50)})

## 7 · Export INT8 quantized TFLite model

In [ ]:
OUTPUT_DIR = Path('/content/hapticway_output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Export INT8 TFLite. The 'data' param provides calibration images for INT8 quantisation.
export_path = best.export(
    format='tflite',
    int8=True,
    imgsz=320,
    data=str(yaml_path),
)
print('Exported to:', export_path)

# Ultralytics saves the model inside a _saved_model folder — find the .tflite file
import glob
tflite_files = glob.glob('/content/runs/hapticway/weights/best_saved_model/*.tflite')
print('TFLite files found:', tflite_files)

# Copy to output dir with our target filename
import shutil as _sh
src_tflite = next(f for f in tflite_files if 'int8' in f)
dst_tflite = OUTPUT_DIR / 'hapticway_custom.tflite'
_sh.copy(src_tflite, dst_tflite)
print('Model size:', dst_tflite.stat().st_size // 1024, 'KB')

In [ ]:
# Write label file for the Flutter app.
# IMPORTANT: YOLOv8 outputs 0-indexed class IDs (0=person, 1=bicycle...).
# The app's postprocess.dart uses labels[classIdx] directly (no +1 offset) for YOLOv8.
# See the integration note in cell 8.

labels_out = OUTPUT_DIR / 'hapticway_labels.txt'
with open(labels_out, 'w') as f:
    for name in CLASSES:
        f.write(name + '\n')

print('Label file:')
print(labels_out.read_text())

## 8 · Download outputs and integrate into the app

Run the cell below to download both files, then follow these steps:

**Files to copy:**
1. `hapticway_custom.tflite` → `assets/models/hapticway_custom.tflite`
2. `hapticway_labels.txt` → `assets/labels/hapticway_labels.txt` (overwrite the placeholder)

**Code changes needed in the Flutter app:**

In `lib/inference/camera_isolate.dart`, find the `// §5.6:` comment and swap the paths:
```dart
final modelData = await rootBundle.load('assets/models/hapticway_custom.tflite');
final labelsRaw = await rootBundle.loadString('assets/labels/hapticway_labels.txt');
```

In `lib/inference/postprocess.dart`, change `labelIdx = classIdx + 1` to `labelIdx = classIdx`
(YOLOv8 is 0-indexed — there is no background class at index 0).

In `pubspec.yaml`, add: `- assets/models/hapticway_custom.tflite`

Then run `flutter run` and test on device.

In [ ]:
from google.colab import files

files.download(str(OUTPUT_DIR / 'hapticway_custom.tflite'))
files.download(str(OUTPUT_DIR / 'hapticway_labels.txt'))

## 9 · Verify output tensor format

In [ ]:
import numpy as np
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path=str(OUTPUT_DIR / 'hapticway_custom.tflite'))
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('=== INPUT ===')
for d in input_details:
    print(f"  [{d['index']}] shape={d['shape']}  dtype={d['dtype']}")

print('=== OUTPUT ===')
for d in output_details:
    print(f"  [{d['index']}] shape={d['shape']}  dtype={d['dtype']}")

# Expected for YOLOv8n with 7 classes at imgsz=320:
#   Input  shape=[1, 320, 320, 3]  dtype=uint8
#   Output shape=[1, 11, 2100]     dtype=float32
#   where 11 = 4 bbox coords + 7 class scores
#         2100 = 40×40 + 20×20 + 10×10 anchor points

# Dummy inference
dummy = np.zeros((1, 320, 320, 3), dtype=np.uint8)
interpreter.set_tensor(input_details[0]['index'], dummy)
interpreter.invoke()
out = interpreter.get_tensor(output_details[0]['index'])
print(f'\nDummy output shape: {out.shape}')
print('Dummy inference OK.')